In [24]:
@file:DependsOn("lib/build/libs/lib.jar")

# Tic-Tac-Toe Library Demo

This notebook demonstrates usage and key concepts featured in the *Tic-Tac-Toe* library

The library provides a clean separation between **game logic** (library) and **user interface** (your application).

## Key Concepts

### Mandatory for integration
- **Mark**: X, O, or EMPTY
- **Game**: Main controller for game flow
- **GameConfig**: Configuration class for game engine
- **AIPlayer**: Intelligent opponent using minimax algorithm

### AI Algorithm
The AI uses **minimax with alpha-beta pruning**:
- Explores all possible future moves
- Chooses the move that maximizes its chances
- Uses **transposition tables** to cache positions
- Adapts search depth based on game phase

### Imports

In [25]:
import org.jetbrains.kotlinx.tictactoe.engine.*
import org.jetbrains.kotlinx.tictactoe.engine.cache.*
import org.jetbrains.kotlinx.tictactoe.model.*
import org.jetbrains.kotlinx.tictactoe.model.enums.*

println("✅ Library imported successfully!")

✅ Library imported successfully!


# Basic Engine integration
- This Kotlin snippet demonstrates how to configure and initialize a game with optional AI players.

```kotlin
/*
    AIPlayer map represents a map with AI players; can be 0, 1, or 2.
    Create AIPlayer if needed:
        aiPlayers[Mark.O | Mark.X] = AIPlayer(Mark.O | Mark.X)

    By default, players are human if `aiPlayers` does not have an AIPlayer assigned.
*/

val aiPlayers = mutableMapOf<Mark, AIPlayer>()

val config = GameConfig()

val game = Game(
    playerX = "player 1",
    playerO = "player 2",
    aiPlayers = aiPlayers,
    cnfg = config
)
```


### Game controls

The following Kotlin snippet demonstrates the main flow of a game, including checking for termination, handling AI moves, and persisting results.

```kotlin
// Get a list of raw cells ; cell type is Mark
var cells = game.getRawBoard()

// Print formatted board
game.displayBoard()

// Termination condition
val over = game.isOver

// If there is an AI player
val ai = game.hasAI()

// Move action
// Retrieve move for AI player
val move = game.autoMove()

// makeMove accepts integers 0 - 8
val success = game.makeMove(move)
// If not successful -> the cell is occupied

// Game result
println("\n" + game.resultMessage())

// Persist game cache
game.end()
```


### CLI demo integration is inside [CLI - APP](./app/src/main/kotlin/org/jetbrains/kotlinx/tictactoe/main.kt)

------------


# Featuring library *examples* and *concepts* used

## Example 1: Board Operations

The `Board` class manages the 3x3 grid with positions **0–8**:

| 0 | 1 | 2 |
|---|---|---|
| 3 | 4 | 5 |
| 6 | 7 | 8 |


Basic operations:
- Create a board
- Place marks
- Check available moves
- Detect wins

In [26]:
val board = Board()

println("New board:")
println("  Available moves: ${board.availableMoves()}")
println("  Is full: ${board.isFull()}")
println()

board.placeMark(4, Mark.X)
board.placeMark(0, Mark.O)

println("After moves:")
println("  Available moves: ${board.availableMoves()}")
println("  Winner: ${board.getWinner() ?: "None yet"}")

val cells = board.getCells()
for (i in 0..2) {
    val row = cells.subList(i * 3, i * 3 + 3)
        .map { when(it) { Mark.X -> "X"; Mark.O -> "O"; else -> " " } }
    println("  ${row.joinToString(" | ")}")
    if (i < 2) println("  ---------")
}

New board:
  Available moves: [0, 1, 2, 3, 4, 5, 6, 7, 8]
  Is full: false

After moves:
  Available moves: [1, 2, 3, 5, 6, 7, 8]
  Winner: None yet
  O |   |  
  ---------
    | X |  
  ---------
    |   |  


## Example 2: Human vs Human Game

The `Game` class manages:
- Player turns
- Move validation
- Win detection
- Game state

Let's simulate a simple game between two human players.

In [27]:
val game = Game(
    playerX = "Alice",
    playerO = "Bob"
)

println("Game created: ${game.currentPlayerName} starts")
println(game.displayBoard())

val moves = listOf(
    4 to "Alice",  // Center
    0 to "Bob",    // Corner
    2 to "Alice",  // Corner
    1 to "Bob",    // Top middle
    6 to "Alice"   // Bottom left - Alice wins!
)

for ((position, player) in moves) {
    println("\n$player plays position $position")
    game.makeMove(position)
    println(game.displayBoard())

    if (game.isOver) {
        println("Game Over!")
        break
    }
}

println("\nResult: ${game.resultMessage()}")

Game created: Alice starts

   |   |   
---+---+---
   |   |   
---+---+---
   |   |   


Alice plays position 4

   |   |   
---+---+---
   | X |   
---+---+---
   |   |   


Bob plays position 0

 O |   |   
---+---+---
   | X |   
---+---+---
   |   |   


Alice plays position 2

 O |   | X 
---+---+---
   | X |   
---+---+---
   |   |   


Bob plays position 1

 O | O | X 
---+---+---
   | X |   
---+---+---
   |   |   


Alice plays position 6

 O | O | X 
---+---+---
   | X |   
---+---+---
 X |   |   

Game Over!

Result: X wins!


## Example 3: Adding AI Players

The AI uses **Minimax Algorithm**:

### How Minimax Works
1. **Simulate** all possible future moves
2. **Evaluate** end positions (win/lose)
3. **Backpropagate** scores up the game tree
4. **Choose** the move with best outcome

### Optimizations
- **Alpha-Beta Pruning**: Skip branches that can't improve result
- **Transposition Tables**: Cache evaluated positions
- **Zobrist Hashing**: Fast position comparison

The AI **never loses** with optimal play.

In [28]:
val aiPlayer = AIPlayer(Mark.O)

val vsAIGame = Game(
    playerX = "Alice",
    playerO = "AI",
    aiPlayers = mapOf(Mark.O to aiPlayer),
    cnfg = GameConfig()
)

println("Human vs AI Game")
println("Alice (X) vs AI (O)")
println(vsAIGame.displayBoard())

println("\nAlice plays position 4 (center)")
vsAIGame.makeMove(4)
println(vsAIGame.displayBoard())

if (vsAIGame.hasAI()) {
    val aiMove = vsAIGame.autoMove()
    println("\nAI plays position $aiMove")
    println(vsAIGame.displayBoard())
}
var aliceMove = 0
game.board.availableMoves().forEach { it ->  aliceMove = it}
println("\nAlice plays position ${aliceMove} (corner)")

vsAIGame.makeMove(aliceMove)
println(vsAIGame.displayBoard())

if (vsAIGame.hasAI()) {
    val aiMove = vsAIGame.autoMove()
    println("\nAI plays position $aiMove")
    println(vsAIGame.displayBoard())
}

println("\nGame continues... AI makes strategic decisions based on minimax!")

Human vs AI Game
Alice (X) vs AI (O)

   |   |   
---+---+---
   |   |   
---+---+---
   |   |   


Alice plays position 4 (center)

   |   |   
---+---+---
   | X |   
---+---+---
   |   |   


AI plays position 0

 O |   |   
---+---+---
   | X |   
---+---+---
   |   |   


Alice plays position 8 (corner)

 O |   |   
---+---+---
   | X |   
---+---+---
   |   | X 


AI plays position 2

 O |   | O 
---+---+---
   | X |   
---+---+---
   |   | X 


Game continues... AI makes strategic decisions based on minimax!


- Current library setup features a random heuristic factor. Which makes the system play *greedy* at certain times.
- Greed factor resonates in the random integer which gets multiplied with heuristic value.
- This means that AI will shift between playing defensive to attacking.

## Example 4: AI vs AI Battle

Watch two AI players compete to see optimal strategies in action.

With perfect play from both sides, the game should always end in a draw.

In [29]:
println("⚔️ EXAMPLE 5: AI vs AI Battle\n")

val aiX = AIPlayer(Mark.X, Zobrist(seed = 100L))
val aiO = AIPlayer(Mark.O, Zobrist(seed = 200L))

val aiGame = Game(
    playerX = "AI-Alpha",
    playerO = "AI-Beta",
    aiPlayers = mapOf(Mark.X to aiX, Mark.O to aiO),
    ttPath = null,
    cnfg = GameConfig(aiDepthStart = 10, aiDepthMid = 12, aiDepthFinish = 15)
)

println("Battle begins!")
println(aiGame.displayBoard())

var moveCount = 0
while (!aiGame.isOver && moveCount < 9) {
    moveCount++
    val currentPlayer = if (aiGame.currentMark == Mark.X) "AI-Alpha" else "AI-Beta"
    val move = aiGame.autoMove()
    println("\nMove $moveCount: $currentPlayer plays position $move")
    println(aiGame.displayBoard())
}

println("🏁 ${aiGame.resultMessage()}")
println("(With perfect play, result should be a draw)")

⚔️ EXAMPLE 5: AI vs AI Battle

Battle begins!

   |   |   
---+---+---
   |   |   
---+---+---
   |   |   


Move 1: AI-Alpha plays position 6

   |   |   
---+---+---
   |   |   
---+---+---
 X |   |   


Move 2: AI-Beta plays position 4

   |   |   
---+---+---
   | O |   
---+---+---
 X |   |   


Move 3: AI-Alpha plays position 0

 X |   |   
---+---+---
   | O |   
---+---+---
 X |   |   


Move 4: AI-Beta plays position 3

 X |   |   
---+---+---
 O | O |   
---+---+---
 X |   |   


Move 5: AI-Alpha plays position 5

 X |   |   
---+---+---
 O | O | X 
---+---+---
 X |   |   


Move 6: AI-Beta plays position 1

 X | O |   
---+---+---
 O | O | X 
---+---+---
 X |   |   


Move 7: AI-Alpha plays position 7

 X | O |   
---+---+---
 O | O | X 
---+---+---
 X | X |   


Move 8: AI-Beta plays position 8

 X | O |   
---+---+---
 O | O | X 
---+---+---
 X | X | O 


Move 9: AI-Alpha plays position 2

 X | O | X 
---+---+---
 O | O | X 
---+---+---
 X | X | O 

🏁 Draw!
(With perfect p

## Example 5: Zobrist Hashing

Zobrist hashing converts board positions into unique 64-bit integers.

**How it works:**
1. Pre-generate random numbers for each (position, mark) pair
2. XOR them together based on board state
3. Same position → same hash (O(1) lookup)
4. Different positions → different hashes (collision-resistant)

In [30]:
// Board helper method
fun displayBoard(board: Board){
    val marks = board.getCells()
    var counter = 1
    marks.forEach { it ->
        if (counter % 3 == 0){
            print(" " + it.name + "\n")
        }else{
            if (it.name.length == 1){
                print("    "+ it.name + "\t|")
            }else {
                print(" " + it.name + "\t|")
            }
            }
        counter++
    }
    println()
}

println("🔐 EXAMPLE 6: Zobrist Hashing\n")

val zobrist = Zobrist(seed = 12345L)

val board1 = Board()
board1.placeMark(0, Mark.X)
board1.placeMark(4, Mark.O)

val board2 = Board()
board2.placeMark(4, Mark.O)
board2.placeMark(0, Mark.X)

println("Board 1:")
displayBoard(board1)

println("Board 2:")
displayBoard(board2)

val hash1 = zobrist.hash(board1, Mark.X)
val hash2 = zobrist.hash(board2, Mark.X)

println("Board 1 hash: $hash1")
println("Board 2 hash: $hash2")
println("Hashes equal: ${hash1 == hash2} ✅")
println("\nSame position → same hash → O(1) cache lookup!")

🔐 EXAMPLE 6: Zobrist Hashing

Board 1:
    X	| EMPTY	| EMPTY
 EMPTY	|    O	| EMPTY
 EMPTY	| EMPTY	| EMPTY

Board 2:
    X	| EMPTY	| EMPTY
 EMPTY	|    O	| EMPTY
 EMPTY	| EMPTY	| EMPTY

Board 1 hash: -279747870425641514
Board 2 hash: -279747870425641514
Hashes equal: true ✅

Same position → same hash → O(1) cache lookup!


## Example 6: Transposition Table

The transposition table (TT) caches evaluated positions:

**Benefits:**
- Avoid recalculating positions
- Persistent across games
- Shared between AI players

**Structure:**

In [31]:
println("💾 EXAMPLE 6: Transposition Table\n")

val sharedCache = mutableMapOf<Long, TTEntry>()

val cachedAI = AIPlayer(Mark.X, Zobrist(42L))
cachedAI.attachCache(sharedCache)

println("Initial cache size: ${sharedCache.size}")

val analysisBoard = Board()
analysisBoard.placeMark(4, Mark.O)

println("\nAI analyzing board where O took center...")
val move = cachedAI.chooseMove(analysisBoard, depth = 12)

println("Best move found: $move")
println("Cache size after analysis: ${sharedCache.size}")
println("These are now cached for instant lookup in future games.")

println("\nAnalyzing same position again...")
val start = System.currentTimeMillis()
cachedAI.chooseMove(analysisBoard, depth = 12)
val time = System.currentTimeMillis() - start
println("Time with cache: ${time}ms")

💾 EXAMPLE 6: Transposition Table

Initial cache size: 0

AI analyzing board where O took center...
Best move found: 8
Cache size after analysis: 1497
These are now cached for instant lookup in future games.

Analyzing same position again...
Time with cache: 0ms


## Example 7: Adaptive Depth by Phase

The AI adjusts Minimax search depth based on game phase:

| Phase | Depth | Rationale |
|-------|-------|-----------|
| START | Low (6-10) | Many options, less critical |
| MID | Medium (10-15) | Balanced approach |
| FINISH | High (15-20) | Few options, critical decisions |


**Configurable via:** *GameConfig*

In [32]:
println("🎯 EXAMPLE 7: Adaptive Depth\n")

val adaptiveConfig = GameConfig(
    aiDepthStart = 6,
    aiDepthMid = 10,
    aiDepthFinish = 18
)

println("Configuration:")
println("  START phase: depth ${adaptiveConfig.aiDepthStart}")
println("  MID phase: depth ${adaptiveConfig.aiDepthMid}")
println("  FINISH phase: depth ${adaptiveConfig.aiDepthFinish}")

val adaptiveAI = AIPlayer(Mark.O)
val adaptiveGame = Game(
    playerX = "Human",
    playerO = "Adaptive-AI",
    aiPlayers = mapOf(Mark.O to adaptiveAI),
    ttPath = null,
    cnfg = adaptiveConfig
)


🎯 EXAMPLE 7: Adaptive Depth

Configuration:
  START phase: depth 6
  MID phase: depth 10
  FINISH phase: depth 18


## Example 8: Tournament Simulation

Run multiple games to verify AI behavior.

**Expected Result:** With perfect play from both AIs, all games should be draws.

In [33]:
println("🏁 EXAMPLE 8: Tournament\n")

fun runTournament(games: Int, depth: Int): Map<String, Int> {
    var xWins = 0
    var oWins = 0
    var draws = 0

    val config = GameConfig(
        aiDepthStart = depth,
        aiDepthMid = depth,
        aiDepthFinish = depth
    )

    repeat(games) {
        val tournamentAiX = AIPlayer(Mark.X)
        val tournamentAiO = AIPlayer(Mark.O)

        val tournamentGame = Game(
            playerX = "AI-X",
            playerO = "AI-O",
            aiPlayers = mapOf(Mark.X to tournamentAiX, Mark.O to tournamentAiO),
            ttPath = null,
            cnfg = config
        )

        while (!tournamentGame.isOver) {
            tournamentGame.autoMove()
        }

        when (tournamentGame.winner) {
            Mark.X -> xWins++
            Mark.O -> oWins++
            else -> draws++
        }
    }

    return mapOf("X" to xWins, "O" to oWins, "Draw" to draws)
}

println("Running tournament: 10 games at depth 10...\n")
val results = runTournament(10, 10)

println("Tournament Results:")
println("  X wins: ${results["X"]}")
println("  O wins: ${results["O"]}")
println("  Draws: ${results["Draw"]}")

🏁 EXAMPLE 8: Tournament

Running tournament: 10 games at depth 10...

Tournament Results:
  X wins: 2
  O wins: 0
  Draws: 8


## Summary

This notebook demonstrated:

### ✅ Core Features
- Board management and move validation
- Game state with phase transitions
- Intelligent AI using minimax algorithm
- Win detection for all patterns

### ✅ Performance Optimizations
- Zobrist hashing for O(1) position lookup
- Alpha-beta pruning (~50% fewer nodes)
- Adaptive depth by game phase

### ✅ Architecture Highlights
- **Separation of Concerns**: Library is UI-agnostic
- **Testability**: Pure functions, comprehensive tests
- **Extensibility**: Easy to add new player types
- **Performance**: Optimized algorithms with caching

### 🎯 Key Takeaways
1. Minimax guarantees optimal play
2. Caching dramatically improves performance
3. Phase-based depth improves UX
4. Clean architecture enables reuse
